In [1]:
%pip install -q firebase-admin

import os
import json
from datetime import datetime

import numpy as np
import pandas as pd
import tensorflow as tf
import joblib

import firebase_admin
from firebase_admin import credentials, firestore

Note: you may need to restart the kernel to use updated packages.


In [2]:
SERVICE_ACCOUNT_PATH = "pregnancy-belt-fb490-firebase-adminsdk-fbsvc-e5b424a068.json"  # adjust if name differs

cred = credentials.Certificate(SERVICE_ACCOUNT_PATH)
firebase_admin.initialize_app(cred)

db = firestore.client()
print("Firestore client initialized.")

Firestore client initialized.


In [4]:
OUTPUT_DIR = "model_outputs_oversample"

model_path = os.path.join(OUTPUT_DIR, "best_model.keras")
dynamic_scaler_path = os.path.join(OUTPUT_DIR, "dynamic_scaler.pkl")
static_scaler_path  = os.path.join(OUTPUT_DIR, "static_scaler.pkl")

print("Model path exists:", os.path.exists(model_path))
print("Dynamic scaler path exists:", os.path.exists(dynamic_scaler_path))
print("Static scaler path exists:", os.path.exists(static_scaler_path))

model = tf.keras.models.load_model(model_path)
dynamic_scaler = joblib.load(dynamic_scaler_path)
static_scaler  = joblib.load(static_scaler_path)

print("Model and scalers loaded.")

Model path exists: True
Dynamic scaler path exists: True
Static scaler path exists: True
Model and scalers loaded.


d:\Projects\Maternal-Care\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [5]:
WINDOW_SIZE = 240

DYNAMIC_FEATURES = ["FHR", "UC"]

STATIC_FEATURES_CONT = ["pH", "pCO2", "BE", "Gest. weeks", "Weight(g)", "Age"]
STATIC_FEATURES_BIN  = ["Seizures", "Sex", "Diabetes", "Hypertension"]
STATIC_FEATURES = STATIC_FEATURES_CONT + STATIC_FEATURES_BIN

ADD_MISSING_INDICATORS = True

CLASS_NAMES = {
    0: "critical (0-3)",
    1: "moderate (4-6)",
    2: "best (7-10)",
}

In [6]:
ANN_DB_PATH = "ann_db.csv"

# If not yet downloaded in this notebook, re-download:
!curl -LO https://raw.githubusercontent.com/Rishabh672003/maternal-health/refs/heads/main/ann_db.csv

def load_clinical_metadata(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    first_col = df.columns[0]
    df = df.rename(columns={first_col: "variable"}).set_index("variable")
    meta = df.T
    meta.index.name = "patient_id"
    meta.index = meta.index.astype(str)
    meta.columns = [c.strip() for c in meta.columns]
    return meta

meta_df = load_clinical_metadata(ANN_DB_PATH)
print("meta_df shape:", meta_df.shape)
display(meta_df.head())

static_cols = [
    "pH", "pCO2", "BE", "Apgar1", "Apgar5", "Seizures",
    "Gest. weeks", "Weight(g)", "Sex", "Age", "Diabetes", "Hypertension",
]

clinical_medians = meta_df[static_cols].median().to_dict()
print("Clinical medians:", clinical_medians)

meta_df shape: (552, 12)


  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100  33467 100  33467   0      0  42633      0                              0
100  33467 100  33467   0      0  42617      0                              0
100  33467 100  33467   0      0  42609      0                              0


,pH,pCO2,BE,Apgar1,Apgar5,Seizures,Gest. weeks,Weight(g),Sex,Age,Diabetes,Hypertension
patient_id,,,,,,,,,,,,
1275,7.20,7.6,-6.6,10.0,10.0,0.0,41.0,4000.0,1.0,19.0,0.0,0.0
1092,7.26,7.1,-4.4,9.0,9.0,0.0,41.0,2980.0,1.0,28.0,0.0,0.0
1180,7.07,9.1,-12.0,8.0,8.0,0.0,39.0,3390.0,2.0,35.0,0.0,0.0
1099,7.14,8.3,-9.0,4.0,7.0,0.0,40.0,3050.0,2.0,29.0,0.0,0.0
1046,7.29,5.9,-5.6,9.0,10.0,0.0,40.0,3250.0,2.0,21.0,0.0,0.0


Clinical medians: {'pH': 7.25, 'pCO2': 7.0, 'BE': -5.6, 'Apgar1': 9.0, 'Apgar5': 9.0, 'Seizures': 0.0, 'Gest. weeks': 40.0, 'Weight(g)': 3410.0, 'Sex': 1.0, 'Age': 30.0, 'Diabetes': 0.0, 'Hypertension': 0.0}


In [7]:
def build_fb_dataframe_from_sensor_snapshot(sensor_data: dict, patient_id="FB_001") -> pd.DataFrame:
    """
    sensor_data: dict with nested structure:
      {
        "AD8232": {"ecgHrBpm": ..., ...},
        "FSR": {"fsrAU": ...},
        ...
      }
    Returns: fb_df_final with shape (240, 16)
    """
    # Map Firebase fields to model fields
    fhr_val = sensor_data.get("AD8232", {}).get("ecgHrBpm", 0)
    uc_val  = sensor_data.get("FSR", {}).get("fsrAU", 0)

    window_len = WINDOW_SIZE

    fhr_list    = [fhr_val] * window_len
    uc_list     = [uc_val]  * window_len
    seconds_arr = list(range(window_len))
    pid_list    = [patient_id] * window_len

    fb_df = pd.DataFrame({
        "patient_id": pid_list,
        "seconds": seconds_arr,
        "FHR": fhr_list,
        "UC": uc_list,
    })

    # Add clinical medians as static columns
    for col, val in clinical_medians.items():
        fb_df[col] = val

    standard_column_order = [
        "patient_id", "seconds", "FHR", "UC",
        "pH", "pCO2", "BE", "Apgar1", "Apgar5", "Seizures",
        "Gest. weeks", "Weight(g)", "Sex", "Age", "Diabetes", "Hypertension",
    ]

    fb_df_final = fb_df[standard_column_order]
    return fb_df_final

In [8]:
# Quick check: list some sensor_logs_10s documents
logs_ref = db.collection("sensor_logs_10s")
docs = list(logs_ref.limit(3).stream())
print("Found", len(docs), "sensor_logs_10s docs.")
for d in docs:
    print("Doc id:", d.id)
    print("Keys:", list(d.to_dict().keys()))
    break  # just show first

Found 3 sensor_logs_10s docs.
Doc id: GozQMHlrfwolGGWZypou
Keys: ['windowStartTs', 'samples', 'windowEndTs']


In [9]:
def extract_sensor_snapshot_from_log(log_doc: dict) -> dict:
    samples = log_doc.get("samples", [])
    if not samples:
        raise ValueError("No samples in log_doc")

    mid = len(samples) // 2
    mid_sample = samples[mid]
    sensor_data = mid_sample.get("Sensor_data", {})
    return sensor_data

In [10]:
def run_inference_on_fb_df(fb_df_final: pd.DataFrame):
    # 1. Dynamic features
    dynamic_data = fb_df_final[DYNAMIC_FEATURES].values  # (240, 2)

    # 2. Static base features
    static_data_base = fb_df_final[STATIC_FEATURES].iloc[0:1].values  # (1, 10)

    # 3. Add missing indicators if used during training
    if ADD_MISSING_INDICATORS:
        n_cont = len(STATIC_FEATURES_CONT)  # 6
        missing_indicators = np.zeros((1, n_cont))
        static_data_enriched = np.concatenate([static_data_base, missing_indicators], axis=1)
    else:
        static_data_enriched = static_data_base

    # 4. Scale
    dynamic_scaled = dynamic_scaler.transform(dynamic_data)
    X_time_inf = dynamic_scaled.reshape(1, WINDOW_SIZE, len(DYNAMIC_FEATURES))

    static_scaled = static_scaler.transform(static_data_enriched)
    X_static_inf = static_scaled  # (1, 16)

    # 5. Predict
    probs = model.predict([X_time_inf, X_static_inf], verbose=0)[0]  # (num_classes,)
    pred_idx = int(np.argmax(probs))
    pred_label = CLASS_NAMES.get(pred_idx, f"class_{pred_idx}")

    probs_list = probs.tolist()
    critical_prob = float(probs_list[0]) if len(probs_list) > 0 else 0.0
    moderate_prob = float(probs_list[1]) if len(probs_list) > 1 else 0.0
    best_prob     = float(probs_list[2]) if len(probs_list) > 2 else 0.0

    return {
        "predictedStatus": pred_label,
        "criticalProb": critical_prob,
        "moderateProb": moderate_prob,
        "bestProb": best_prob,
    }

In [11]:
def run_inference_for_latest_log():
    # 1. Get latest sensor_logs_10s document by windowEndTs
    logs_ref = db.collection("sensor_logs_10s")
    query_ref = logs_ref.order_by("windowEndTs", direction=firestore.Query.DESCENDING).limit(1)
    docs = list(query_ref.stream())

    if not docs:
        print("No sensor_logs_10s documents found.")
        return

    doc = docs[0]
    log_doc = doc.to_dict()
    log_id  = doc.id

    print("Using log doc:", log_id)

    # 2. Extract snapshot from log
    sensor_snapshot = extract_sensor_snapshot_from_log(log_doc)

    # 3. Build fb_df_final
    fb_df_final = build_fb_dataframe_from_sensor_snapshot(sensor_snapshot, patient_id="FB_001")
    print("fb_df_final shape:", fb_df_final.shape)

    # 4. Run inference
    result = run_inference_on_fb_df(fb_df_final)
    print("Inference result:", result)

    # 5. Prepare prediction document
    pred_doc = {
        "windowDocId": log_id,
        "patientId": "FB_001",
        "windowStartTs": log_doc.get("windowStartTs"),
        "windowEndTs": log_doc.get("windowEndTs"),

        "predictedStatus": result["predictedStatus"],
        "criticalProb": result["criticalProb"],
        "moderateProb": result["moderateProb"],
        "bestProb": result["bestProb"],

        "modelVersion": "ctu-chb-v1",
        "createdAt": datetime.utcnow().isoformat() + "Z",
    }

    # 6. Write to Firestore predictions collection
    preds_ref = db.collection("predictions")
    preds_ref.document(log_id).set(pred_doc)

    print("Wrote prediction document with id:", log_id)
    return pred_doc

In [12]:
pred_doc = run_inference_for_latest_log()
pred_doc

Using log doc: kSKkVteegCy54MEjOYkV
fb_df_final shape: (240, 16)
Inference result: {'predictedStatus': 'critical (0-3)', 'criticalProb': 0.5141909718513489, 'moderateProb': 0.4858090281486511, 'bestProb': 0.0}
Wrote prediction document with id: kSKkVteegCy54MEjOYkV


{'windowDocId': 'kSKkVteegCy54MEjOYkV',
 'patientId': 'FB_001',
 'windowStartTs': '2025-12-15 14:36:38.007 IST',
 'windowEndTs': '2025-12-15 14:36:54.490 IST',
 'predictedStatus': 'critical (0-3)',
 'criticalProb': 0.5141909718513489,
 'moderateProb': 0.4858090281486511,
 'bestProb': 0.0,
 'modelVersion': 'ctu-chb-v1',
 'createdAt': '2026-03-22T11:50:18.133832Z'}